# 09 - Parameter and pixel-metric figure routines

This notebook confirms the parameter-space and per-pixel figure routines of `Ploter` on controlled inputs: spatial parameter maps, GT-vs-pred scatter, absolute-error maps, the active-Gaussian-count agreement map, a per-pixel metric map, and metric histograms. Each routine writes PNGs which are read back and displayed.

Modules exercised: `pipelines.inference_pipeline.plots.Ploter` (`plot_param_maps`, `plot_param_scatter`, `plot_param_error_maps`, `plot_active_count_map`, `plot_pixel_metric_map`, `plot_metric_histograms`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Controlled parameter cubes

Two Gaussian slots over an `H x W` grid. The GT means form smooth spatial gradients; the prediction adds small noise. Slot 1 is a placeholder on the right third (amplitude 0, NaN mu / sigma) so the active-count map has something to show. A pixel-MSE map is derived for the per-pixel-map and histogram routines.

In [ ]:
import tempfile
import matplotlib.image as mpimg
from pipelines.inference_pipeline.plots import Ploter

H, W, K = 24, 32, 2
rng = np.random.default_rng(0)
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')

params_gt   = np.zeros((3 * K, H, W), dtype=np.float32)
params_gt[0] = 1.0; params_gt[1] = -5.0 + 8.0 * xx / (W - 1); params_gt[2] = 2.0
params_gt[3] = 0.7; params_gt[4] =  6.0 - 8.0 * yy / (H - 1); params_gt[5] = 1.5

placeholder = xx >= (2 * W // 3)
params_gt[3][placeholder] = 0.0
params_gt[4][placeholder] = np.nan
params_gt[5][placeholder] = np.nan

params_pred = params_gt.copy()
for ch in (1, 2, 4, 5):
    params_pred[ch] = params_gt[ch] + 0.15 * rng.standard_normal((H, W))
params_pred[3][placeholder] = 0.0  # predict the placeholders correctly

pixel_mse = (0.01 + 0.05 * rng.random((H, W))).astype(np.float32)
out_dir   = Path(tempfile.mkdtemp(prefix='nb09_'))
plotter   = Ploter(cmap='jet', err_cmap='magma', normalize=False, fig_dpi=110, save_dpi=110)
print('figures ->', out_dir)


## Spatial parameter maps (pred vs GT)

`plot_param_maps` writes a pred and a GT map for each `(slot, parameter)` pair on a shared colour scale. We display the slot-0 mean maps side by side.

In [ ]:
map_paths = plotter.plot_param_maps(
    params_pred=params_pred, params_gt=params_gt, n_gaussians=K,
    out_dir=out_dir / 'maps', az_offset=0, rg_offset=0,
)
pred_mu = next(p for p in map_paths if p.name == 'g1_mu_pred.png')
gt_mu   = next(p for p in map_paths if p.name == 'g1_mu_gt.png')

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
for ax, p in zip(axes, [gt_mu, pred_mu]):
    ax.imshow(mpimg.imread(p)); ax.axis('off'); ax.set_title(p.name)
fig.tight_layout()
plt.show()


## GT-vs-pred scatter

`plot_param_scatter` writes a scatter with an identity line and an R-squared annotation per `(slot, parameter)`. Since the prediction tracks GT closely the points should hug the diagonal. We display the slot-0 mean scatter.

In [ ]:
scatter_paths = plotter.plot_param_scatter(
    params_pred=params_pred, params_gt=params_gt, n_gaussians=K,
    out_dir=out_dir / 'scatter',
)
mu_scatter = next(p for p in scatter_paths if p.name == 'g1_mu.png')
fig, ax = plt.subplots(figsize=(5.5, 5.0))
ax.imshow(mpimg.imread(mu_scatter)); ax.axis('off')
ax.set_title('plot_param_scatter: g1 mu')
plt.show()


## Absolute-error maps and active-count agreement

`plot_param_error_maps` writes `|pred - gt|` maps per parameter; `plot_active_count_map` writes an agreement map colouring each pixel by whether the predicted number of active Gaussians matches GT. We display the slot-0 mean error map and the agreement map.

In [ ]:
err_paths = plotter.plot_param_error_maps(
    params_pred=params_pred, params_gt=params_gt, n_gaussians=K,
    out_dir=out_dir / 'err', az_offset=0, rg_offset=0,
)
count_paths = plotter.plot_active_count_map(
    params_pred=params_pred, params_gt=params_gt, n_gaussians=K,
    out_dir=out_dir / 'count', az_offset=0, rg_offset=0,
)
mu_err = next(p for p in err_paths if p.name == 'g1_mu.png')
agree  = next(p for p in count_paths if 'agreement' in p.name)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
axes[0].imshow(mpimg.imread(mu_err)); axes[0].axis('off'); axes[0].set_title('|d mu| g1')
axes[1].imshow(mpimg.imread(agree));  axes[1].axis('off'); axes[1].set_title('active-count agreement')
fig.tight_layout()
plt.show()


## Per-pixel metric map and histograms

`plot_pixel_metric_map` renders a single metric field (here pixel MSE in log scale); `plot_metric_histograms` writes one histogram per supplied metric array. We display the MSE map and its histogram.

In [ ]:
mse_map_path = plotter.plot_pixel_metric_map(
    metric_map=pixel_mse, title='Per-pixel MSE (demo)', label='MSE',
    out_path=out_dir / 'pixel_mse.png', az_offset=0, rg_offset=0,
    cmap='magma', log=True,
)
hist_paths = plotter.plot_metric_histograms({'pixel_mse': pixel_mse}, out_dir / 'hist')

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.0))
axes[0].imshow(mpimg.imread(mse_map_path)); axes[0].axis('off'); axes[0].set_title('pixel MSE map (log)')
axes[1].imshow(mpimg.imread(hist_paths[0])); axes[1].axis('off'); axes[1].set_title('pixel MSE histogram')
fig.tight_layout()
plt.show()


## Expected visual outcome

The slot-0 mean maps should show matching left-to-right gradients for GT and prediction; the scatter should cluster on the identity line with a high R-squared; the error map should be low and roughly uniform; the active-count agreement map should be mostly the correct colour with the placeholder third handled consistently; and the pixel-MSE map and histogram should render the small synthetic error distribution. Successful display of every panel confirms the figure routines accept the pipeline's array contracts.